# 07b. Random Forest Hyperparameter Tuning - Automobile Loan Default Prediction

Tunes the Random Forest and compares it with the untuned baseline. Both are evaluated with the same stratified 5-fold cross-validation on the training split, at the default 0.5 threshold. The held-out test rows are discarded at load time, and no operating threshold is chosen in this notebook.


## 1. Setup


In [1]:
import os
import sys

project_root = os.path.abspath("..") if os.path.basename(os.getcwd()) == "notebooks" else os.path.abspath(".")
if project_root not in sys.path:
    sys.path.insert(0, project_root)

import pandas as pd
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold

from src.config import RANDOM_STATE, TARGET
from src.data.data_cleaning import clean_and_split
from src.evaluation.metrics import evaluate_model, log_result_to_csv
from src.models.random_forest import build_random_forest_pipeline

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 1000)

train_df, _ = clean_and_split()  # held-out test rows are discarded here
X_train, y_train = train_df.drop(columns=[TARGET]), train_df[TARGET]

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
results_dir = "experiments/results" if os.path.exists("experiments/results") else "../experiments/results"
experiments_path = os.path.join(results_dir, "experiments.csv")

print("X_train shape:", X_train.shape)
print("Default rate (train):", round(y_train.mean(), 4))


X_train shape: (95372, 40)
Default rate (train): 0.081


## 2. Untuned Baseline (same folds)

The baseline is re-evaluated here under the same folds as the tuned model, so the comparison is fair. It uses the untuned Random Forest from `src/models/random_forest.py` at the default 0.5 threshold.


In [2]:
baseline_pipeline = build_random_forest_pipeline()

baseline_results = evaluate_model(baseline_pipeline, X_train, y_train, model_name="random_forest", cv=5)

for key, value in baseline_results.items():
    print(f"{key}: {value}")

log_result_to_csv(baseline_results, experiments_path)


model_name: random_forest
cv_folds: 5
threshold: 0.5
roc_auc_mean: 0.7740530162105269
roc_auc_std: 0.00399387962965928
pr_auc_mean: 0.3575537792965477
pr_auc_std: 0.00932055501149178
recall_mean: 0.16338598521052367
precision_mean: 0.6320702646114715
f1_mean: 0.25954910625331074
confusion_matrix: [[86910, 738], [6462, 1262]]
fit_seconds: 48.4


,model_name,cv_folds,threshold,roc_auc_mean,roc_auc_std,pr_auc_mean,pr_auc_std,recall_mean,precision_mean,f1_mean,fit_seconds,tn,fp,fn,tp
0,gaussian_naive_bayes,5,0.5,0.696815,0.003703,0.176248,0.008833,0.479027,0.169532,0.250394,17.15,69518,18130,4024,3700
1,XGBoost Baseline,5,0.5,0.763093,0.004383,0.261403,0.009897,0.022009,0.621743,0.042488,39.12,87542,106,7554,170
2,decision_tree,5,0.5,0.708183,0.003783,0.180718,0.004465,0.606682,0.151856,0.242232,15.97,61284,26364,3038,4686
3,decision_tree,5,0.5,0.708183,0.003783,0.180718,0.004465,0.606682,0.151856,0.242232,17.56,61284,26364,3038,4686
4,LightGBM Baseline,5,0.5,0.748190,0.003018,0.228391,0.008387,0.644226,0.167997,0.266493,22.52,63002,24646,2748,4976
5,random_forest,5,0.5,0.774053,0.003994,0.357554,0.009321,0.163386,0.632070,0.259549,48.40,86910,738,6462,1262


## 3. Search Space

Each list includes the baseline value, so the untuned configuration is reachable by the search.

- `n_estimators`: number of trees (baseline 300).
- `max_depth`: tree depth limit (baseline `None`, unlimited).
- `min_samples_leaf`: minimum applicants per leaf (baseline 1).
- `max_features`: features considered per split (baseline `sqrt`).
- `class_weight`: class weighting (baseline `balanced`).


In [3]:
param_distributions = {
    "model__n_estimators": [200, 300, 500],
    "model__max_depth": [None, 8, 12, 16, 20],
    "model__min_samples_leaf": [1, 5, 10, 20, 50],
    "model__max_features": ["sqrt", "log2", 0.3, 0.5],
    "model__class_weight": [None, "balanced", "balanced_subsample"],
}


## 4. Randomized Search

30 sampled configurations, each evaluated with the same stratified 5-fold split and scored by ROC-AUC. Each fold refits the full pipeline, including the feature-selection step, so the search cannot leak information from validation folds. Expect 35-45 minutes.


In [ ]:
search = RandomizedSearchCV(
    baseline_pipeline,
    param_distributions=param_distributions,
    n_iter=30,
    scoring="roc_auc",
    cv=skf,
    random_state=RANDOM_STATE,
    n_jobs=1,
    refit=True,
    verbose=1,
)
search.fit(X_train, y_train)

print("Best CV ROC-AUC:", round(search.best_score_, 4))
print("Best parameters:", search.best_params_)


Fitting 5 folds for each of 30 candidates, totalling 150 fits


## 5. Tuned Model (same folds)


In [ ]:
tuned_results = evaluate_model(
    search.best_estimator_,
    X_train,
    y_train,
    model_name="random_forest_tuned",
    cv=5,
)

for key, value in tuned_results.items():
    print(f"{key}: {value}")

log_result_to_csv(tuned_results, experiments_path)


model_name: random_forest_tuned
cv_folds: 5
threshold: 0.5
roc_auc_mean: 0.7698177887093822
roc_auc_std: 0.004141125885795309
pr_auc_mean: 0.33880401728021026
pr_auc_std: 0.009195984950402428
recall_mean: 0.28456763418683034
precision_mean: 0.38068323200848253
f1_mean: 0.3256503267970598
confusion_matrix: [[84073, 3575], [5526, 2198]]
fit_seconds: 68.65


,model_name,cv_folds,threshold,roc_auc_mean,roc_auc_std,pr_auc_mean,pr_auc_std,recall_mean,precision_mean,f1_mean,fit_seconds,tn,fp,fn,tp
0,XGBoost Baseline,5,0.5,0.763093,0.004383,0.261403,0.009897,0.022009,0.621743,0.042488,39.12,87542,106,7554,170
1,decision_tree,5,0.5,0.708183,0.003783,0.180718,0.004465,0.606682,0.151856,0.242232,15.97,61284,26364,3038,4686
2,decision_tree,5,0.5,0.708183,0.003783,0.180718,0.004465,0.606682,0.151856,0.242232,17.56,61284,26364,3038,4686
3,gaussian_naive_bayes,5,0.5,0.696815,0.003703,0.176248,0.008833,0.479027,0.169532,0.250394,16.14,69518,18130,4024,3700
4,random_forest,5,0.5,0.774053,0.003994,0.357554,0.009321,0.163386,0.632070,0.259549,43.26,86910,738,6462,1262
5,random_forest_tuned,5,0.5,0.769818,0.004141,0.338804,0.009196,0.284568,0.380683,0.325650,68.65,84073,3575,5526,2198


## 6. Baseline vs Tuned

The tuned model counts as better only if its ROC-AUC gain is larger than the baseline's ROC-AUC standard deviation across folds.


In [ ]:
comparison = pd.DataFrame([baseline_results, tuned_results]).drop(columns=['confusion_matrix'])
comparison = comparison[
    ['model_name', 'roc_auc_mean', 'roc_auc_std', 'pr_auc_mean', 'pr_auc_std',
     'recall_mean', 'precision_mean', 'f1_mean']
]

roc_gain = tuned_results['roc_auc_mean'] - baseline_results['roc_auc_mean']
print(comparison.to_string(index=False))
print()
print(f"ROC-AUC gain: {roc_gain:.4f}  (baseline std: {baseline_results['roc_auc_std']:.4f})")
print("Tuned is clearly better" if roc_gain > baseline_results['roc_auc_std'] else "No clear improvement over baseline")


         model_name  roc_auc_mean  roc_auc_std  pr_auc_mean  pr_auc_std  recall_mean  precision_mean  f1_mean
      random_forest      0.774053     0.003994     0.357554    0.009321     0.163386        0.632070 0.259549
random_forest_tuned      0.769818     0.004141     0.338804    0.009196     0.284568        0.380683 0.325650

ROC-AUC gain: -0.0042  (baseline std: 0.0040)
No clear improvement over baseline


## 7. Save Search Results and Best Parameters


In [ ]:
cv_results = pd.DataFrame(search.cv_results_)
cv_results.to_csv(os.path.join(results_dir, "random_forest_random_search.csv"), index=False)

best_params_df = pd.DataFrame(
    [{"parameter": k, "value": v} for k, v in search.best_params_.items()]
)
best_params_df.to_csv(os.path.join(results_dir, "random_forest_best_parameters.csv"), index=False)
print(best_params_df.to_string(index=False))


              parameter    value
    model__n_estimators      500
model__min_samples_leaf        1
    model__max_features     sqrt
       model__max_depth       20
    model__class_weight balanced


**Finding:** Tuning did not beat the untuned baseline on ranking. Baseline ROC-AUC is 0.774 +/- 0.004 and the tuned model's is 0.770 +/- 0.004, a gain of -0.004, which is within one standard deviation. PR-AUC also falls, from 0.358 to 0.339. The best sampled configuration (500 trees, max depth 20, leaf size 1, sqrt features, balanced weights) is close to the baseline setting, so the search points to the baseline already sitting near the best of the sampled space. The exact baseline setting was not among the 30 sampled configurations, so this is not an exhaustive search.

At the default 0.5 threshold, the tuned model has higher recall (0.285 vs. 0.163) and lower precision (0.381 vs. 0.632). That is a shift in the operating point, not better ranking.

**Decision:** Keep the untuned baseline configuration as the Random Forest model, since it has the best ROC-AUC and PR-AUC. The best sampled configuration is saved to `experiments/results/random_forest_best_parameters.csv` for reference only, and is not adopted. `src/models/random_forest.py` is not changed. The operating threshold remains a separate team decision, applied to all models.
